In [0]:
from utils.logger import get_logger

CATALOG = "acme_catalog"
GOLD_SCHEMA = "gold"
GOLD = f"{CATALOG}.{GOLD_SCHEMA}"

logger = get_logger("gold_marts")

In [0]:
spark.sql(f"""
    create or replace table {GOLD}.agg_sales_by_month
    comment 'Gold KPI: monthly sales aggregated by division, country and category'
    as
    select
        dd.year, dd.quarter, dd.month, dd.month_name,
        cast(dd.year * 10000 + dd.month * 100 + 1 as int) as date_id,
        dc.division_id, dc.country, dp.category_id,
        cast(sum(fol.revenue) as decimal(14,2)) as total_revenue,
        cast(sum(fol.cost) as decimal(14,2)) as total_cost,
        cast(sum(fol.margin) as decimal(14,2)) as total_margin,
        count(distinct fo.order_id) as order_count,
        count(distinct fo.customer_sk) as distinct_customers
    from {GOLD}.fact_order_lines fol
    join {GOLD}.fact_orders fo on fol.order_id = fo.order_id
    join {GOLD}.dim_date dd on fo.date_id = dd.date_id
    join {GOLD}.dim_customers dc on fo.customer_sk = dc.customer_sk and dc.is_current = true
    join {GOLD}.dim_products dp on fol.product_sk = dp.product_sk and dp.is_current = true
    group by dd.year, dd.quarter, dd.month, dd.month_name, dc.division_id, dc.country, dp.category_id
""")

row_count = spark.table(f"{GOLD}.agg_sales_by_month").count()
logger.info(f"agg_sales_by_month rebuilt: {row_count} rows")

In [0]:
spark.sql(f"""
    create or replace table {GOLD}.agg_customer_growth_by_month
    comment 'Gold KPI: monthly customer growth with new, active and YTD counts'
    as
    with monthly_customers as (
        select
            dd.year, dd.quarter, dd.month, dd.month_name, fo.customer_sk,
            min(dd.full_date) over (
                partition by fo.customer_sk order by dd.full_date
                rows between unbounded preceding and unbounded following
            ) as first_order_date
        from {GOLD}.fact_orders fo
        join {GOLD}.dim_date dd on fo.date_id = dd.date_id
    ),
    monthly_agg as (
        select
            year, quarter, month, month_name,
            count(distinct customer_sk) as active_customers,
            count(distinct case
                when date_format(first_order_date, 'yyyyMM') = concat(year, lpad(month, 2, '0'))
                then customer_sk
            end) as new_customers
        from monthly_customers
        group by year, quarter, month, month_name
    )
    select
        year, quarter, month, month_name,
        cast(year * 10000 + month * 100 + 1 as int) as date_id,
        new_customers, active_customers,
        sum(new_customers) over (
            partition by year order by month
            rows between unbounded preceding and current row
        ) as ytd_customers
    from monthly_agg
""")

row_count = spark.table(f"{GOLD}.agg_customer_growth_by_month").count()
logger.info(f"agg_customer_growth_by_month rebuilt: {row_count} rows")